# Notebook 4 — CMF Unlearning: cmf_static vs cmf_dynamic (3:7 Cross-Class Split)

**Paper:** *An Illusion of Unlearning?* (Gao et al., AISTATS 2026 · arXiv 2604.08271)

**Prerequisite:** Notebook 1 output attached. Set `CKPT_DATASET_DIR` below.

**Two CMF algorithms** (same 3-metric eval as NB3):

| Strategy | Description |
|----------|-------------|
| `cmf_static` | Paper Algorithm 1: W rebuilt via closed-form CMF each epoch, then **frozen** |
| `cmf_dynamic` | AlternatingCMF: W reset to CMF at start of each round, then **gradient-updated** in Phase 2 |

**Experiment matrix:**
```
for base_method in [scrub, neggrad_plus, random_label, salun]:
  for classifier_strategy in [cmf_static, cmf_dynamic]:
    for mean_source in [train, retain]:
      for seed in [0, 1, 2]:
```

**Paper's CMF formula (eq.3, eq.7):** μ_c = mean over ALL training samples of class c (D_r ∪ D_f).  
`mean_source='train'` reproduces this exactly. `mean_source='retain'` is an ablation variant.  

**Known deviation preserved:** `recompute_cmf` L2-normalizes features before averaging (paper uses raw features).  
Paper Figure 6 caption acknowledges this as 'an additional normalization step'. Not changed here.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, random, argparse, collections, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__, '  CUDA:', torch.cuda.is_available())

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ══ SET THIS to the Kaggle dataset mount path from Notebook 1 ══
CKPT_DATASET_DIR = '/kaggle/input/regun-notebook1'  # ← EDIT

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/regun_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/regun/regun_config.json',
    f'{CKPT_DATASET_DIR}/regun/regun_config.json',
]
config_path = CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p; CKPT_ROOT_NB1 = os.path.dirname(_p); break
assert config_path, 'regun_config.json not found. Check CKPT_DATASET_DIR.'

with open(config_path) as f: CFG = json.load(f)
TEST_MODE         = CFG['TEST_MODE']
TEST_FRACTION     = CFG['TEST_FRACTION']
_MODE_TAG         = CFG['_MODE_TAG']
DATASET           = CFG['DATASET']
ARCH              = CFG['ARCH']
IS_VIT            = CFG['IS_VIT']
NUM_CLASSES       = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES = CFG['CLASS_LABEL_NAMES']
SPLIT_SEEDS       = CFG['SPLIT_SEEDS']
FORGET_FRACTION   = CFG['FORGET_FRACTION']
PRETRAIN_LR       = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS   = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS       = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']
_TOTAL     = {DATASET: CFG['TOTAL']}
_PER_CLASS = {DATASET: CFG['PER_CLASS']}

_old_root = CFG['CKPT_ROOT']
def _repath(p): return p.replace(_old_root, CKPT_ROOT_NB1)
CKPT_PRETRAIN      = _repath(CFG['CKPT_PRETRAIN'])
CKPT_PRETRAIN_FLAT = _repath(CFG.get('CKPT_PRETRAIN_FLAT',
    CFG['CKPT_PRETRAIN'].replace('.pt', '_encoder_flat.pt')))
SPLIT_DIR     = _repath(CFG['SPLIT_DIR'])

DATA_PATH     = '/kaggle/working/data'
CKPT_ROOT_NB4 = '/kaggle/working/checkpoints/regun_nb4'
os.makedirs(CKPT_ROOT_NB4, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

MAX_EPOCHS = 1 if TEST_MODE else 50
UNLEARN_BS = 8 if TEST_MODE else 128
print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}  MAX_EPOCHS={MAX_EPOCHS}')
print(f'  [{"OK" if os.path.exists(CKPT_PRETRAIN) else "MISSING"}] CKPT_PRETRAIN: {CKPT_PRETRAIN}')

In [ ]:
from utils import get_dataset, get_model, test, SubSet
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw): return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

_CMF_LR = {
    'scrub':        {'cifar10': 5e-3, 'cifar100': 5e-3, 'tinyimagenet': 5e-3},
    'neggrad_plus': {'cifar10': 1e-4, 'cifar100': 1e-4, 'tinyimagenet': 3e-5},
    'random_label': {'cifar10': 1e-4, 'cifar100': 2e-3, 'tinyimagenet': 1e-2},
    'salun':        {'cifar10': 2e-4, 'cifar100': 2e-3, 'tinyimagenet': 1e-2},
}
_STATIC_EPOCHS = {'scrub': 3, 'neggrad_plus': 3, 'random_label': 4, 'salun': 4}
if TEST_MODE: _STATIC_EPOCHS = {k: 1 for k in _STATIC_EPOCHS}

if TEST_MODE:
    _DYN_CFG = {'rounds': 1, 't_theta': 1, 't_w': 1}  # 1*(1+1)=2 <= MAX_EPOCHS
else:
    _DYN_CFG = {'rounds': 5, 't_theta': 5, 't_w': 5}  # 5*(5+5)=50

print(f'cmf_static epochs: {_STATIC_EPOCHS}')
print(f'cmf_dynamic: {_DYN_CFG}  epoch-equiv: '
      f'{_DYN_CFG["rounds"]}*({_DYN_CFG["t_theta"]}+{_DYN_CFG["t_w"]})='
      f'{_DYN_CFG["rounds"]*(_DYN_CFG["t_theta"]+_DYN_CFG["t_w"])} <= {MAX_EPOCHS}')

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5, min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=1.0, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=True, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='regun_cmf',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

def load_cmf_checkpoint(model, ckpt_path, device):
    """
    Load a checkpoint into a ModelModule regardless of whether it was saved
    from a ModelModule (encoder.* keys) or a bare ResNet (flat keys).
    If the checkpoint has flat keys (e.g. conv1.weight) but the model expects
    encoder.* keys, the keys are remapped automatically — no need to re-run NB1.
    """
    raw_sd = torch.load(ckpt_path, map_location=device)
    model_keys = set(model.state_dict().keys())
    ckpt_keys  = set(raw_sd.keys())

    # Already matches — load directly
    if ckpt_keys <= model_keys or not ckpt_keys.isdisjoint(model_keys):
        missing, unexpected = model.load_state_dict(raw_sd, strict=False)
        if missing:   print(f'  [load_cmf_checkpoint] missing  : {missing[:3]}...')
        if unexpected: print(f'  [load_cmf_checkpoint] unexpected: {unexpected[:3]}...')
        return model

    # Flat keys (conv1.weight) → remap to encoder.conv1.weight
    remapped = {}
    for k, v in raw_sd.items():
        if 'encoder.' + k in model_keys:
            remapped['encoder.' + k] = v
        else:
            remapped[k] = v  # keep as-is (e.g. CMFweights.weight)

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    if missing:    print(f'  [load_cmf_checkpoint] still missing after remap: {missing[:5]}')
    if unexpected: print(f'  [load_cmf_checkpoint] unexpected after remap: {unexpected[:3]}')
    return model

print('load_cmf_checkpoint helper defined.')


In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _ss(ds, frac, seed=42):
        labels = (ds.targets if hasattr(ds,'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds,'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_cls = collections.defaultdict(list)
        for i,l in enumerate(labels): by_cls[int(l)].append(i)
        kept = []
        for c in sorted(by_cls):
            pool = by_cls[c]; rng.shuffle(pool)
            kept.extend(pool[:max(1,math.ceil(len(pool)*frac))])
        sub = torch.utils.data.Subset(ds, kept)
        bt = ds.targets if hasattr(ds,'targets') else [ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [bt[i] for i in kept]
        return sub
    dataset_train = _ss(dataset_train, TEST_FRACTION)
    dataset_test  = _ss(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=UNLEARN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256,len(dataset_test)), num_workers=2, pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(
    dataset_train, batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(dataset_test, **TEST_KW)

splits = {}
for seed in SPLIT_SEEDS:
    sf = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    assert os.path.exists(sf), f'Split file missing: {sf}'
    with open(sf) as f: splits[seed] = json.load(f)
    print(f'Seed {seed}: forget={splits[seed]["n_forget"]}  retain={splits[seed]["n_retain"]}')

args_pt = make_args(unlearn_method='CMF_pre_train', remove_FC=True, CMFClassifier=True)
orig_model = get_model(args_pt, device)
load_cmf_checkpoint(orig_model, CKPT_PRETRAIN, device)
orig_model.eval()
if hasattr(orig_model, 'recompute_cmf'):
    orig_model.recompute_cmf(train_loader, device=device)
print('\n── Θ_o test accuracy ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')

## Three-Metric Eval Harness (Index-Based)

Identical to NB1/NB2/NB3. Output, Linear Probe, NCC — all over sample index sets.

In [ ]:
def eval_output_on_indices(model, dataset, indices, device, batch_size=256):
    if len(indices) == 0: return float('nan')
    loader = torch.utils.data.DataLoader(
        SubSet(dataset, indices), batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
    return correct / max(1, total)


@torch.no_grad()
def _extract_features(model, loader, device):
    model.eval(); Xs, ys = [], []
    for x, y in loader:
        x = x.to(device)
        f = model.extract_features(x) if hasattr(model,'extract_features') else model(x)
        Xs.append(f.cpu()); ys.append(y)
    return torch.cat(Xs,0).float(), torch.cat(ys,0).long()


def _test_split_30_70(yte_numpy):
    rng = random.Random(0)
    by_cls = collections.defaultdict(list)
    for i, l in enumerate(yte_numpy): by_cls[int(l)].append(i)
    fgt, ret = [], []
    for c in sorted(by_cls):
        pool = list(by_cls[c]); rng.shuffle(pool)
        nf = max(1, round(len(pool)*0.30))
        fgt.extend(pool[:nf]); ret.extend(pool[nf:])
    return ret, fgt


def eval_probe_on_indices(model, dataset_train, dataset_test,
                          retain_indices, forget_indices,
                          device, num_classes,
                          probe_epochs=100, probe_lr=0.01, batch_size=256):
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    Xtr, ytr = _extract_features(model, torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True), device)
    Xte, yte = _extract_features(model, torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True), device)
    head = nn.Linear(Xtr.size(1), num_classes).to(device)
    opt  = optim.SGD(head.parameters(), lr=probe_lr, momentum=0.9)
    ldr  = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(probe_epochs):
        head.train()
        for bx, by in ldr:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()
    head.eval()
    with torch.no_grad(): pred = head(Xte.to(device)).argmax(1).cpu()
    ret_idx, fgt_idx = _test_split_30_70(yte.numpy())
    def _acc(idxs): return float((pred[idxs]==yte[idxs]).float().mean()) if idxs else float('nan')
    for p in model.parameters(): p.requires_grad_(True)
    return _acc(ret_idx), _acc(fgt_idx)


def eval_ncc_on_indices(model, dataset_train, dataset_test,
                        retain_indices, forget_indices,
                        device, num_classes, batch_size=256):
    model.eval()
    Xtr, ytr = _extract_features(model, torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True), device)
    Xte, yte = _extract_features(model, torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True), device)
    Xtr_n = F.normalize(Xtr, dim=1)
    cmeans = torch.zeros(num_classes, Xtr.size(1))
    for c in range(num_classes):
        m = (ytr==c)
        if m.any(): cmeans[c] = Xtr_n[m].mean(0)
    cmeans_n = F.normalize(cmeans, dim=1)
    Xte_n = F.normalize(Xte, dim=1)
    with torch.no_grad(): pred = (Xte_n @ cmeans_n.t()).argmax(1)
    ret_idx, fgt_idx = _test_split_30_70(yte.numpy())
    def _acc(idxs): return float((pred[idxs]==yte[idxs]).float().mean()) if idxs else float('nan')
    return _acc(ret_idx), _acc(fgt_idx)


def eval_three_metrics(model, dataset_train, dataset_test,
                       retain_indices, forget_indices,
                       device, num_classes,
                       run_probe=True, run_ncc=True, probe_epochs=100):
    out_r = eval_output_on_indices(model, dataset_train, retain_indices, device)
    out_f = eval_output_on_indices(model, dataset_train, forget_indices,  device)
    if run_probe and not TEST_MODE:
        pr_r, pr_f = eval_probe_on_indices(
            model, dataset_train, dataset_test, retain_indices, forget_indices,
            device, num_classes, probe_epochs=probe_epochs)
    else: pr_r = pr_f = float('nan')
    if run_ncc and not TEST_MODE:
        ncc_r, ncc_f = eval_ncc_on_indices(
            model, dataset_train, dataset_test, retain_indices, forget_indices,
            device, num_classes)
    else: ncc_r = ncc_f = float('nan')
    return dict(output_retain_acc=out_r, output_forget_acc=out_f,
                probe_retain_acc=pr_r,  probe_forget_acc=pr_f,
                ncc_retain_acc=ncc_r,   ncc_forget_acc=ncc_f)


print('Three-metric eval harness defined.')

## C. CMF Algorithm Implementations

### cmf_static — reproduces paper Algorithm 1
Every epoch: `recompute_cmf(mu_loader)` → W frozen → encoder updated via base method loss.  
`mean_source='train'` is paper-faithful (eq.3 averages all D=D_r∪D_f).  
`mean_source='retain'` is an ablation variant.

### cmf_dynamic — new AlternatingCMF
```
warm-start W via CMF (mean_source loader)
for r in range(rounds):
    # Reset W to paper's CMF solution at start of every Phase 1
    model.recompute_cmf(mu_loader)  # paper-faithful reset
    # Phase 1: W frozen, update encoder for t_theta steps
    # Phase 2: encoder frozen, update W via CE on retain set for t_w steps
Budget: rounds * (t_theta + t_w) <= MAX_EPOCHS
```

In [ ]:
def cmf_static_unlearn(
    model, device, retain_loader, forget_loader, train_loader,
    args, epochs, base_method, mean_source,
    forget_indices, retain_indices, dataset_train, dataset_test,
):
    """
    cmf_static: paper Algorithm 1.
    Every epoch: recompute_cmf(mu_loader) -> freeze W -> update encoder.

    NOTE: recompute_cmf L2-normalizes features before averaging (known deviation
    from paper eq.7 which averages raw features). Preserved unchanged here.
    mean_source='train' is paper-faithful (eq.3 uses full D=D_r∪D_f).
    """
    assert hasattr(model, 'recompute_cmf'), 'cmf_static requires ModelModule with recompute_cmf'

    # Paper-faithful: use full train set; 'retain' is ablation
    mu_loader = train_loader if mean_source == 'train' else retain_loader

    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.lr, momentum=0.9, weight_decay=5e-4, nesterov=True)

    epoch_logs = []
    forget_iterator = iter(forget_loader) if forget_loader else None

    for epoch in range(1, epochs + 1):
        # Rebuild W (closed-form CMF), then frozen (it's a buffer, no requires_grad)
        model.eval(); model.recompute_cmf(mu_loader, device=device); model.train()

        total_loss = total_n = 0
        for x, y in retain_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            W_fixed = model.CMFweights.weight.detach()
            f_r = model.extract_features(x)
            z_r = model._preprocess_feats_for_cmf(f_r)
            logits_r = z_r @ W_fixed.t()
            loss = F.cross_entropy(logits_r, y)

            if forget_loader and base_method in ('neggrad_plus','scrub','random_label','salun'):
                try: x_f, y_f = next(forget_iterator)
                except StopIteration: forget_iterator = iter(forget_loader); x_f, y_f = next(forget_iterator)
                x_f, y_f = x_f.to(device), y_f.to(device)
                f_f = model.extract_features(x_f)
                z_f = model._preprocess_feats_for_cmf(f_f)
                logits_f = z_f @ W_fixed.t()
                if base_method == 'random_label':
                    y_rand = torch.randint(0, NUM_CLASSES, y_f.shape, device=device)
                    y_rand[y_rand == y_f] = (y_rand[y_rand == y_f] + 1) % NUM_CLASSES
                    loss = loss + F.cross_entropy(logits_f, y_rand)
                else:
                    loss = loss - F.cross_entropy(logits_f, y_f)

            loss.backward()
            if getattr(args, 'grad_norm_clip', None):
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_norm_clip)
            optimizer.step()
            total_loss += loss.item() * x.size(0); total_n += x.size(0)

        model.eval(); model.recompute_cmf(mu_loader, device=device)
        ra = eval_output_on_indices(model, dataset_train, retain_indices, device)
        fa = eval_output_on_indices(model, dataset_train, forget_indices,  device)
        print(f'  [cmf_static ep {epoch}/{epochs}] R={ra:.4f} F={fa:.4f} loss={total_loss/max(1,total_n):.4f}')
        epoch_logs.append({'epoch': epoch, 'retain': ra, 'forget': fa})
        model.train()

    model.eval(); model.recompute_cmf(mu_loader, device=device)
    return model, epoch_logs


print('cmf_static_unlearn defined.')

In [ ]:
def cmf_dynamic_unlearn(
    model, device, retain_loader, forget_loader, train_loader,
    args, base_method, mean_source,
    forget_indices, retain_indices, dataset_train, dataset_test,
    rounds=5, t_theta=5, t_w=5,
    phase2_data='retain_only',
    w_init_mode='cmf',
):
    """
    cmf_dynamic: AlternatingCMF.

    KEY DESIGN: W is reset to the paper's closed-form CMF solution at the start
    of EVERY Phase 1. This ensures Phase 1 always starts from the analytical
    CMF baseline, not from the gradient-modified W of the previous round.
    Phase 2 then refines W from that CMF baseline via gradient CE.

    mean_source controls mu_loader for both the pre-loop warm-start AND
    the per-round CMF reset:
      'train'  → full train set (paper-faithful: eq.3 uses D=D_r∪D_f)
      'retain' → retain set only (ablation variant)

    Budget: rounds*(t_theta+t_w) <= MAX_EPOCHS (logged for auditability).

    NOTE: L2-normalize-before-averaging in recompute_cmf is preserved unchanged
    (known deviation from paper eq.7; Figure 6 caption acknowledges it).
    """
    assert hasattr(model, 'CMFweights'), 'cmf_dynamic requires ModelModule with CMFweights'

    total_epoch_equiv = rounds * (t_theta + t_w)
    assert total_epoch_equiv <= MAX_EPOCHS, (
        f'Budget violation: {rounds}*({t_theta}+{t_w})={total_epoch_equiv} > MAX_EPOCHS={MAX_EPOCHS}')
    print(f'  [cmf_dynamic] budget: {rounds}*({t_theta}+{t_w})={total_epoch_equiv} <= {MAX_EPOCHS} ✓')
    print(f'  [cmf_dynamic] per_round_cmf_reset=True  reset_source={mean_source}')

    # mu_loader: paper-faithful='train', ablation='retain'
    mu_loader = train_loader if mean_source == 'train' else retain_loader

    # ── Pre-loop warm-start W ─────────────────────────────────────────
    model.eval()
    if w_init_mode == 'cmf':
        model.recompute_cmf(mu_loader, device=device)
        print(f'  [cmf_dynamic] pre-loop warm-start: CMF (mean_source={mean_source})')
    else:
        with torch.no_grad():
            torch.nn.init.normal_(model.CMFweights.weight, 0, 0.01)
            model.CMFweights.weight.copy_(F.normalize(model.CMFweights.weight, dim=1))
        print(f'  [cmf_dynamic] pre-loop warm-start: RANDOM (w_init_mode=random)')
    print(f'  NOTE: per-round reset always uses mu_loader (mean_source={mean_source}), '
          f'overriding w_init_mode after round 1')

    # Make W a real trainable parameter (CMFWeights stores it as a buffer)
    W_data = model.CMFweights.weight.data.clone()
    W_param = nn.Parameter(W_data.clone())
    del model.CMFweights._buffers['weight']
    model.CMFweights.register_parameter('weight', W_param)

    encoder_params = [p for n, p in model.named_parameters()
                      if 'CMFweights' not in n and p.requires_grad]
    w_params = [model.CMFweights.weight]

    opt_theta = optim.SGD(encoder_params, lr=args.lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
    opt_w     = optim.SGD(w_params, lr=args.lr * 0.1, momentum=0.9, weight_decay=0.0)

    round_logs = []
    forget_iterator = iter(forget_loader) if forget_loader else None

    for r in range(1, rounds + 1):
        print(f'\n  [cmf_dynamic] ── Round {r}/{rounds} ──────────────────────')

        # ── Per-round CMF reset (paper-faithful) ──────────────────────
        # Rebuilds W from the analytical CMF formula before every Phase 1.
        # recompute_cmf writes via .copy_() — works on both buffer and Parameter.
        model.eval()
        model.recompute_cmf(mu_loader, device=device)

        # ── Phase 1: encoder updates, W frozen ────────────────────────
        model.CMFweights.weight.requires_grad_(False)
        model.train()
        for t in range(t_theta):
            for x, y in retain_loader:
                x, y = x.to(device), y.to(device)
                opt_theta.zero_grad()
                W_fixed = model.CMFweights.weight.detach()
                z_r = model._preprocess_feats_for_cmf(model.extract_features(x))
                loss = F.cross_entropy(z_r @ W_fixed.t(), y)

                if forget_loader and base_method in ('neggrad_plus','scrub','random_label','salun'):
                    try: x_f, y_f = next(forget_iterator)
                    except StopIteration: forget_iterator=iter(forget_loader); x_f,y_f=next(forget_iterator)
                    x_f, y_f = x_f.to(device), y_f.to(device)
                    z_f = model._preprocess_feats_for_cmf(model.extract_features(x_f))
                    logits_f = z_f @ W_fixed.t()
                    if base_method == 'random_label':
                        y_rand = torch.randint(0, NUM_CLASSES, y_f.shape, device=device)
                        y_rand[y_rand==y_f] = (y_rand[y_rand==y_f]+1)%NUM_CLASSES
                        loss = loss + F.cross_entropy(logits_f, y_rand)
                    else:
                        loss = loss - F.cross_entropy(logits_f, y_f)

                loss.backward()
                if getattr(args,'grad_norm_clip',None):
                    torch.nn.utils.clip_grad_norm_(encoder_params, args.grad_norm_clip)
                opt_theta.step()
                break  # one batch per step

        # ── Phase 2: W gradient updates, encoder frozen ───────────────
        for p in encoder_params: p.requires_grad_(False)
        model.CMFweights.weight.requires_grad_(True)
        model.train()

        p2_loaders = [retain_loader]
        if phase2_data == 'retain_plus_forget' and forget_loader: p2_loaders.append(forget_loader)
        p2_iters = [iter(ldr) for ldr in p2_loaders]

        for t in range(t_w):
            for p2_it in p2_iters:
                try: x_p2, y_p2 = next(p2_it)
                except StopIteration: p2_it=iter(p2_loaders[p2_iters.index(p2_it)]); x_p2,y_p2=next(p2_it)
                x_p2, y_p2 = x_p2.to(device), y_p2.to(device)
                opt_w.zero_grad()
                with torch.no_grad(): f_p2 = model.extract_features(x_p2); z_p2 = model._preprocess_feats_for_cmf(f_p2)
                W = model.CMFweights.weight
                F.cross_entropy(z_p2 @ W.t(), y_p2).backward()
                opt_w.step()
                with torch.no_grad():
                    model.CMFweights.weight.copy_(F.normalize(model.CMFweights.weight.data, dim=1))
                break

        # Unfreeze encoder
        for p in encoder_params: p.requires_grad_(True)
        model.CMFweights.weight.requires_grad_(False)

        # Audit
        model.eval()
        ra = eval_output_on_indices(model, dataset_train, retain_indices, device)
        fa = eval_output_on_indices(model, dataset_train, forget_indices,  device)
        print(f'  [round {r}] output R={ra:.4f} F={fa:.4f}')
        round_logs.append({'round': r, 'retain': ra, 'forget': fa})
        model.train()

    model.eval()
    return model, round_logs


print('cmf_dynamic_unlearn defined.')

## D. Experiment Matrix

Priority: scrub full sub-matrix first, then random_label, then neggrad_plus/salun.

In [ ]:
BASE_METHODS          = ['scrub', 'random_label', 'neggrad_plus', 'salun']
CLASSIFIER_STRATEGIES = ['cmf_static', 'cmf_dynamic']
MEAN_SOURCES          = ['train', 'retain']
W_INIT_MODES          = ['cmf', 'random']  # cmf_dynamic only; isolates pre-loop seed
PHASE2_DATA           = 'retain_only'

_method_key = {
    'scrub': 'scrub', 'neggrad_plus': 'grad_ascent_descent',
    'random_label': 'random_label', 'salun': 'salun'
}

total = (len(BASE_METHODS)*len(CLASSIFIER_STRATEGIES)*len(MEAN_SOURCES)*len(SPLIT_SEEDS)
         + len(BASE_METHODS)*len(MEAN_SOURCES)*len(SPLIT_SEEDS))  # extra w_init=random for dynamic
print(f'Experiment matrix: {total} total runs (incl. w_init=random ablation)')

In [ ]:
ALL_RESULTS = []

for base_method in BASE_METHODS:
    lr = _CMF_LR[base_method][DATASET]

    for classifier_strategy in CLASSIFIER_STRATEGIES:
        w_init_modes_to_run = (W_INIT_MODES if classifier_strategy == 'cmf_dynamic' else ['cmf'])

        for mean_source in MEAN_SOURCES:
            for seed in SPLIT_SEEDS:
                for w_init_mode in w_init_modes_to_run:
                    run_id = (f'{base_method}__{classifier_strategy}__'
                              f'{mean_source}__seed{seed}'
                              + (f'__winit_{w_init_mode}' if classifier_strategy=='cmf_dynamic' else ''))
                    print(f'\n{"#"*65}\n  {run_id}\n{"#"*65}')

                    split = splits[seed]
                    retain_indices = split['retain_indices']
                    forget_indices = split['forget_indices']

                    retain_ds = SubSet(dataset_train, retain_indices)
                    forget_ds = SubSet(dataset_train, forget_indices)
                    retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)
                    forget_loader = torch.utils.data.DataLoader(forget_ds, **LOADER_KW)

                    args_run = make_args(
                        unlearn_method='CMF_pre_train',  # must contain 'CMF' so get_model returns ModelModule
                        epochs_or_steps=_STATIC_EPOCHS.get(base_method, 3),
                        lr=lr, batch_size=UNLEARN_BS,
                        num_retain_samples=len(retain_indices),
                        num_forget_samples=len(forget_indices),
                        unlearn_class=[], remove_FC=True, CMFClassifier=True,
                        grad_norm_clip=1.0,
                    )

                    model = get_model(args_run, device)
                    load_cmf_checkpoint(model, CKPT_PRETRAIN, device)

                    t0 = time.time()
                    try:
                        if classifier_strategy == 'cmf_static':
                            epochs = min(_STATIC_EPOCHS.get(base_method, 3), MAX_EPOCHS)
                            unlearnt, logs = cmf_static_unlearn(
                                model, device, retain_loader, forget_loader, train_loader,
                                args_run, epochs, _method_key[base_method], mean_source,
                                forget_indices, retain_indices, dataset_train, dataset_test)
                        else:
                            unlearnt, logs = cmf_dynamic_unlearn(
                                model, device, retain_loader, forget_loader, train_loader,
                                args_run, _method_key[base_method], mean_source,
                                forget_indices, retain_indices, dataset_train, dataset_test,
                                rounds=_DYN_CFG['rounds'],
                                t_theta=_DYN_CFG['t_theta'],
                                t_w=_DYN_CFG['t_w'],
                                phase2_data=PHASE2_DATA,
                                w_init_mode=w_init_mode)
                    except Exception as e:
                        import traceback; traceback.print_exc()
                        print(f'  ERROR: {e}')
                        ALL_RESULTS.append(dict(
                            base_method=base_method, classifier_strategy=classifier_strategy,
                            mean_source=mean_source, seed=seed, w_init_mode=w_init_mode,
                            **{k: float('nan') for k in [
                                'output_retain_acc','output_forget_acc',
                                'probe_retain_acc','probe_forget_acc',
                                'ncc_retain_acc','ncc_forget_acc']},
                            wall_clock_minutes=float('nan'), error=str(e)))
                        continue

                    elapsed = time.time() - t0

                    ckpt_dir = f'{CKPT_ROOT_NB4}/{base_method}/{classifier_strategy}'
                    os.makedirs(ckpt_dir, exist_ok=True)
                    ckpt_out = (f'{ckpt_dir}/{DATASET}_{ARCH}_{_MODE_TAG}_{mean_source}_seed{seed}'
                                + (f'_winit_{w_init_mode}' if classifier_strategy=='cmf_dynamic' else '')
                                + '.pt')
                    torch.save(unlearnt.state_dict(), ckpt_out)

                    metrics = eval_three_metrics(
                        unlearnt, dataset_train, dataset_test,
                        retain_indices, forget_indices, device, NUM_CLASSES,
                        run_probe=not TEST_MODE, run_ncc=not TEST_MODE)

                    budget_info = {
                        'MAX_EPOCHS': MAX_EPOCHS,
                        'strategy': classifier_strategy,
                        'per_round_cmf_reset': True if classifier_strategy=='cmf_dynamic' else False,
                        'per_round_reset_source': mean_source,
                        'epoch_equiv': (_STATIC_EPOCHS.get(base_method,3) if classifier_strategy=='cmf_static'
                                        else _DYN_CFG['rounds']*(_DYN_CFG['t_theta']+_DYN_CFG['t_w'])),
                    }
                    print(f'  budget_log: {budget_info}')
                    print(f'  → Output  R={metrics["output_retain_acc"]:.4f}  F={metrics["output_forget_acc"]:.4f}')
                    print(f'  → Probe   R={metrics["probe_retain_acc"]:.4f}  F={metrics["probe_forget_acc"]:.4f}')
                    print(f'  → NCC     R={metrics["ncc_retain_acc"]:.4f}  F={metrics["ncc_forget_acc"]:.4f}')
                    print(f'  → wall clock: {elapsed/60:.1f} min')

                    ALL_RESULTS.append(dict(
                        base_method=base_method,
                        classifier_strategy=classifier_strategy,
                        mean_source=mean_source,
                        seed=seed,
                        w_init_mode=w_init_mode,
                        **metrics,
                        wall_clock_minutes=elapsed/60,
                        budget_epoch_equiv=budget_info['epoch_equiv'],
                    ))

print(f'\nAll runs done. {len(ALL_RESULTS)} results collected.')

## E. Results CSV + Pivoted cmf_static vs cmf_dynamic Comparison Table

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS)
csv_path = f'/kaggle/working/results_cmf_{DATASET}_{ARCH}.csv'
results_df.to_csv(csv_path, index=False)
print(f'Full results saved: {csv_path}  shape: {results_df.shape}')
METRIC_COLS = ['output_retain_acc','output_forget_acc',
               'probe_retain_acc','probe_forget_acc',
               'ncc_retain_acc','ncc_forget_acc']

In [ ]:
# ── Pivoted table: cmf_static vs cmf_dynamic side by side ─────────────
# Filter w_init_mode='cmf' for a clean apples-to-apples comparison
_wim_col = results_df['w_init_mode'] if 'w_init_mode' in results_df.columns else pd.Series('cmf', index=results_df.index)
df_main = results_df[_wim_col == 'cmf'].copy()
present = [c for c in METRIC_COLS if c in df_main.columns and df_main[c].notna().any()]

if not present:
    print('WARNING: no non-NaN metric columns — all runs may have errored. Skipping pivot.')
    agg = pd.DataFrame(columns=['base_method','classifier_strategy','mean_source'])
else:
    agg = df_main.groupby(['base_method','classifier_strategy','mean_source'])[present].agg(['mean','std'])
    agg.columns = ['_'.join(c) for c in agg.columns]
    agg = agg.reset_index()

pivot_rows = []
for bm in BASE_METHODS:
    for ms in MEAN_SOURCES:
        row = {'base_method': bm, 'mean_source': ms}
        for strat in ['cmf_static', 'cmf_dynamic']:
            sub = agg[(agg['base_method']==bm) &
                      (agg['classifier_strategy']==strat) &
                      (agg['mean_source']==ms)]
            if len(sub) == 0: continue
            s = sub.iloc[0]
            for col in ['output_retain_acc','output_forget_acc',
                        'probe_retain_acc','probe_forget_acc',
                        'ncc_retain_acc','ncc_forget_acc']:
                mc, sc = f'{col}_mean', f'{col}_std'
                if mc in s.index:
                    mv, sv = s.get(mc, float('nan')), s.get(sc, float('nan'))
                    row[f'{strat}_{col}'] = (
                        f'{mv:.3f}±{sv:.3f}'
                        if not (pd.isna(mv) or pd.isna(sv)) else 'N/A')
        pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows)

print('\n' + '='*90)
print(f'  CMF STRATEGY COMPARISON — {DATASET}/{ARCH}')
print(f'  mean±std over {len(SPLIT_SEEDS)} seeds (w_init_mode=cmf only)')
print('='*90)
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 220)
print(pivot_df.to_string(index=False))

pivot_path = f'/kaggle/working/pivot_cmf_{DATASET}_{ARCH}.csv'
pivot_df.to_csv(pivot_path, index=False)
print(f'\nPivot table saved: {pivot_path}')

In [ ]:
# ── Ablation: cmf_dynamic w_init_mode='cmf' vs 'random' ───────────────
# Since per-round CMF reset overrides the warm-start every round,
# 'cmf' and 'random' w_init_mode should give identical results (sanity check).
df_dyn = results_df[results_df['classifier_strategy']=='cmf_dynamic'].copy()
if len(df_dyn) > 0 and 'w_init_mode' in df_dyn.columns:
    agg_w = df_dyn.groupby(['base_method','mean_source','w_init_mode'])[
        [c for c in ['output_retain_acc','output_forget_acc'] if c in df_dyn.columns]
    ].agg(['mean','std'])
    agg_w.columns = ['_'.join(c) for c in agg_w.columns]
    print('\n=== cmf_dynamic w_init_mode ablation ===')
    print('(If per-round reset works correctly, cmf vs random init should give same results)')
    print(agg_w.round(4).to_string())

## F. Written Summary: Questions (a)–(c)

In [ ]:
def _fv(v):
    try: return f'{float(v):.4f}'
    except: return 'N/A'

def _get(df, base, strat, src, metric):
    sub = df[(df['base_method']==base) & (df['classifier_strategy']==strat) &
             (df['mean_source']==src) & (df.get('w_init_mode','cmf')=='cmf')]
    if len(sub)==0: return float('nan')
    return sub[metric].mean()

lines = []
lines.append('='*72)
lines.append(f'  WRITTEN SUMMARY — ReGUn CMF Ablation  ({DATASET}/{ARCH})')
lines.append(f'  Split: {FORGET_FRACTION*100:.0f}% forget stratified cross-class (3 seeds)')
lines.append('='*72)

lines.append('\n(a) Does cmf_dynamic get HIGHER RETAIN ACCURACY than cmf_static?')
lines.append('    (same base_method, mean_source=retain, Output metric)')
for bm in BASE_METHODS:
    ra_s = _get(results_df, bm, 'cmf_static',  'retain', 'output_retain_acc')
    ra_d = _get(results_df, bm, 'cmf_dynamic', 'retain', 'output_retain_acc')
    diff = ra_d - ra_s if not (pd.isna(ra_s) or pd.isna(ra_d)) else float('nan')
    tag = ('dynamic higher ↑' if (not pd.isna(diff)) and diff > 0.005
           else 'dynamic lower ↓' if (not pd.isna(diff)) and diff < -0.005
           else 'comparable ≈'  if not pd.isna(diff) else 'N/A')
    lines.append(f'  {bm:15s}: static={_fv(ra_s)}  dynamic={_fv(ra_d)}  Δ={_fv(diff)}  → {tag}')

lines.append('\n(b) Does cmf_dynamic reopen the Output-vs-Probe FORGET GAP?')
lines.append('    ("illusion" pattern: output_forget_acc low, probe_forget_acc high)')
for bm in BASE_METHODS:
    for strat in ['cmf_static','cmf_dynamic']:
        out_f  = _get(results_df, bm, strat, 'retain', 'output_forget_acc')
        prb_f  = _get(results_df, bm, strat, 'retain', 'probe_forget_acc')
        gap = prb_f - out_f if not (pd.isna(prb_f) or pd.isna(out_f)) else float('nan')
        tag = ('YES — gap detected ⚠' if (not pd.isna(gap)) and gap > 0.05
               else 'no clear gap ✓' if not pd.isna(gap) else 'N/A (probe not run in TEST_MODE)')
        lines.append(f'  {bm:15s}/{strat:13s}: out_f={_fv(out_f)} prb_f={_fv(prb_f)} gap={_fv(gap)} → {tag}')

lines.append('\n(c) Does mean_source="retain" vs "train" visibly change results at 30% forget?')
for bm in BASE_METHODS:
    for strat in ['cmf_static','cmf_dynamic']:
        ra_t = _get(results_df, bm, strat, 'train',  'output_retain_acc')
        ra_r = _get(results_df, bm, strat, 'retain', 'output_retain_acc')
        fa_t = _get(results_df, bm, strat, 'train',  'output_forget_acc')
        fa_r = _get(results_df, bm, strat, 'retain', 'output_forget_acc')
        dr = ra_r-ra_t if not (pd.isna(ra_r) or pd.isna(ra_t)) else float('nan')
        df_ = fa_r-fa_t if not (pd.isna(fa_r) or pd.isna(fa_t)) else float('nan')
        lines.append(f'  {bm:15s}/{strat:13s}: Δretain={_fv(dr)}  Δforget={_fv(df_)}')

lines.append('\n' + '='*72)
SUMMARY_TEXT = '\n'.join(lines)
print(SUMMARY_TEXT)

summary_path = f'/kaggle/working/cmf_summary_{DATASET}_{ARCH}.txt'
with open(summary_path, 'w') as f: f.write(SUMMARY_TEXT)
print(f'\nSummary saved: {summary_path}')